# 🎯 AI Recruitment: CV vs JD Semantic Matcher (R&D)

Notebook này đã được cập nhật để trích xuất dữ liệu từ CV thật (PDF/Image) và so sánh với JD nhập tay.

## 1. Setup & Config
Cài đặt đường dẫn Tesseract và load model.

In [ ]:
import fitz
import pytesseract
import os
import io
import pandas as pd
import numpy as np
import re
from PIL import Image, ImageEnhance, ImageOps
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# --- CONFIGURATION ---
TESS_PATH = r"C:\Program Files\Tesseract-OCR\tesseract.exe"
pytesseract.pytesseract.tesseract_cmd = TESS_PATH

MODEL_NAME = "paraphrase-multilingual-MiniLM-L12-v2"
print("Loading model...")
model = SentenceTransformer(MODEL_NAME)
print("Model ready.")

d:\Work\DATN\AI\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3129.00it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model ready.


## 2. Robust Extraction Logic
Các hàm hỗ trợ trích xuất văn bản từ PDF/Ảnh với OCR nâng cao.

In [ ]:
def _to_rgb(pix):
    if pix.alpha or (pix.colorspace and pix.colorspace.n > 3):
        return fitz.Pixmap(fitz.csRGB, pix)
    return pix

def run_ocr(pix):
    """OCR với tiền xử lý tối ưu cho tiếng Việt."""
    png = _to_rgb(pix).tobytes("png")
    img = Image.open(io.BytesIO(png)).convert("L")
    img = ImageOps.autocontrast(img)
    w, h = img.size
    img = img.resize((w * 2, h * 2), Image.Resampling.LANCZOS)
    img = ImageEnhance.Sharpness(img).enhance(2.0)
    return pytesseract.image_to_string(img, lang="vie+eng", config="--psm 3").strip()

def extract_text_from_file(file_path):
    ext = file_path.split('.')[-1].lower()
    text = ""
    if ext == 'pdf':
        doc = fitz.open(file_path)
        for page in doc:
            native = page.get_text("text").strip()
            if native:
                text += native + "\n"
            else:
                pix = page.get_pixmap(matrix=fitz.Matrix(400/72, 400/72))
                text += run_ocr(pix) + "\n"
        doc.close()
    elif ext in ['png', 'jpg', 'jpeg']:
        pix = fitz.Pixmap(file_path)
        text = run_ocr(pix)
    return text.strip()

## 3. Matching Logic
Sử dụng Vector Semantic Similarity.

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^\w\s\+\#\.]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

def match_cv_jd(cv_text, jd_text):
    c_cv = clean_text(cv_text)
    c_jd = clean_text(jd_text)
    
    embeddings = model.encode([c_cv, c_jd])
    score = cosine_similarity([embeddings[0]], [embeddings[1]])[0][0]
    return round(float(score) * 100, 2)

## 4. Execution
Nhập JD và chọn CV để so sánh.

In [ ]:
USER_JD = """
Nhập nội dung JD của bạn vào đây.
Ví dụ: Cần tuyển Python Developer, 2 năm kinh nghiệm, biết Docker và AWS.
"""

CV_FOLDER = r"D:\Work\DATN\AI\CV"
files = [f for f in os.listdir(CV_FOLDER) if f.lower().endswith(('.pdf', '.png', '.jpg'))]

print(f"Tìm thấy {len(files)} CV trong thư mục.")

results = []
for filename in files:
    path = os.path.join(CV_FOLDER, filename)
    print(f"Đang xử lý: {filename}...")
    cv_text = extract_text_from_file(path)
    
    if cv_text:
        score = match_cv_jd(cv_text, USER_JD)
        results.append({"File": filename, "Score": score})
    else:
        print(f"❌ Không thể trích xuất text từ {filename}")

df_results = pd.DataFrame(results).sort_values(by="Score", ascending=False)
df_results

Tìm thấy 3 CV trong thư mục.
Đang xử lý: chi-tiet-mau-cv-so-5.pdf...
Đang xử lý: CV.pdf...
Đang xử lý: test1.pdf...


,File,Score
2,test1.pdf,27.72
1,CV.pdf,22.12
0,chi-tiet-mau-cv-so-5.pdf,21.02
